# F2_04 · Validación del dataset final (colelitiasis, La Araucanía)

**Objetivo:** Validar el subconjunto final que se usará en el análisis, generado en `F2_03_transformacion.ipynb`: revisar que cumpla los requisitos del curso para un dataset multivariado y que las variables creadas en la transformación (provincia, grupos de riesgo, flag de colecistectomía) estén bien construidas, sin nulos inesperados ni valores fuera de lo definido.

**Entrada:** `data/processed/egresos_araucania_k80_transformado.csv`

**Salida:** `docs/validacion_colelitiasis.md` (informe de validación)

**Qué hace este notebook:**
1. Reutiliza el validador del proyecto (`validar_dataset.py`) para perfilar el dataset final y revisar los requisitos del curso.
2. Verifica específicamente que el filtro de colelitiasis, la variable de provincia, los grupos de riesgo y el flag de colecistectomía queden correctos.
3. Revisa las filas duplicadas en el contexto de la limitación ya documentada en el README del proyecto.
4. Guarda un informe de validación en `docs/`.

In [1]:
import sys
import pandas as pd
from pathlib import Path

# El notebook está en F2, así que la raíz del proyecto es la carpeta de arriba
RAIZ = Path.cwd()
if RAIZ.name.startswith("F"):
    RAIZ = RAIZ.parent

# validar_dataset.py está en la raíz del proyecto, no en F2, por eso hay que
# agregar la raíz a sys.path para poder importarlo como módulo
sys.path.insert(0, str(RAIZ))
import validar_dataset as vd

ENTRADA = RAIZ / "data" / "processed" / "egresos_araucania_k80_transformado.csv"
SALIDA_INFORME = RAIZ / "docs" / "validacion_colelitiasis.md"

print("¿Existe la entrada?", ENTRADA.exists())
print(ENTRADA)

¿Existe la entrada? True
D:\Sebiche\Magister Unab\Proyecto 1\proyecto-grupo2-mcdi500\data\processed\egresos_araucania_k80_transformado.csv


In [2]:
datos = pd.read_csv(ENTRADA, dtype=str)

# DIAS_ESTADA y CONDICION_EGRESO son numéricas por naturaleza; se leyeron como
# texto para no perder información al guardar el CSV, pero para perfilarlas
# correctamente (y para cualquier análisis posterior) hay que tiparlas.
datos["DIAS_ESTADA"] = pd.to_numeric(datos["DIAS_ESTADA"])
datos["CONDICION_EGRESO"] = pd.to_numeric(datos["CONDICION_EGRESO"])

print("Filas y columnas:", datos.shape)
datos.head()

Filas y columnas: (3789, 23)


,PERTENENCIA_ESTABLECIMIENTO_SALUD,SEXO,GRUPO_EDAD,ETNIA,GLOSA_PAIS_ORIGEN,COMUNA_RESIDENCIA,GLOSA_COMUNA_RESIDENCIA,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA,PREVISION,...,DIAG2,DIAS_ESTADA,CONDICION_EGRESO,GLOSA_INTERV_Q_PPAL,GLOSA_PROCED_PPAL,PROVINCIA_RESIDENCIA,GRUPO_RIESGO_MUJER,GRUPO_RIESGO_ETNIA,GRUPO_RIESGO_40_49,FLAG_COLECISTECTOMIA
0,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09115,Pucón,09,De La Araucanía,1,...,NaN,1,1,"Colecistectomía por videolaparoscopía, proc. c...",NaN,Cautín,False,False,False,True
1,Pertenecientes al Sistema Nacional de Servicio...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09120,Villarrica,09,De La Araucanía,1,...,NaN,3,1,"Colecistectomía por videolaparoscopía, proc. c...",NaN,Cautín,False,False,False,True
2,Pertenecientes al Sistema Nacional de Servicio...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09211,Victoria,09,De La Araucanía,1,...,NaN,4,1,NaN,NaN,Malleco,False,False,False,False
3,Pertenecientes al Sistema Nacional de Servicio...,MUJER,1 a 9,No se identifica con alguna etnia,Chileno,09102,Carahue,09,De La Araucanía,1,...,NaN,4,1,"Colecistectomía por videolaparoscopía, proc. c...",NaN,Cautín,True,False,False,True
4,Pertenecientes al Sistema Nacional de Servicio...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09206,Los Sauces,09,De La Araucanía,99,...,NaN,1,1,"Colecistectomía por videolaparoscopía, proc. c...",NaN,Malleco,False,False,False,True


## Paso 1 · Perfil general y requisitos del curso

Se reutiliza el mismo validador que se usó para el dataset nacional en la Fase 1 (`validar_dataset.py`), aplicado ahora al subconjunto final.

In [3]:
perfil = vd.perfilar(datos)
perfil

,variable,dtype,rol,unicos,pct_nulos
0,PERTENENCIA_ESTABLECIMIENTO_SALUD,str,binaria,2,0.0
1,SEXO,str,binaria,2,0.0
2,GRUPO_EDAD,str,fecha,10,0.0
3,ETNIA,str,binaria,2,0.0
4,GLOSA_PAIS_ORIGEN,str,binaria,2,0.0
5,COMUNA_RESIDENCIA,str,fecha,32,0.0
6,GLOSA_COMUNA_RESIDENCIA,str,alta cardinalidad,32,0.0
7,REGION_RESIDENCIA,str,fecha,1,0.0
8,GLOSA_REGION_RESIDENCIA,str,nominal,1,0.0
9,PREVISION,str,nominal,6,0.0


In [4]:
chequeo, apto = vd.evaluar(datos, perfil, ENTRADA)
print(chequeo.to_string(index=False))
print()
print("¿El conjunto cumple los requisitos mínimos del curso?", apto)

estado                                       requisito                                                           detalle
    OK                             Al menos 2000 filas                                                        3789 filas
    OK                            Al menos 12 columnas                                                       23 columnas
    OK             Combina al menos 3 roles analíticos binaria, fecha, nominal, alta cardinalidad, discreta, texto libre
    OK                  Al menos una variable numérica                               1 numéricas (continuas o discretas)
    OK                Al menos una variable categórica                  14 categóricas (nominales, binarias u ordinales)
    OK                  Presencia de valores faltantes                                     máximo 100.0% en una variable
 AVISO         Ninguna variable sobre 60% de faltantes                                                     máximo 100.0%
    OK                Incluye al

## Paso 2 · Validaciones específicas de la transformación

Estas revisiones son propias de este notebook, no genéricas del curso: comprueban que el filtro y las variables creadas en `F2_03_transformacion.ipynb` hayan quedado bien construidas.

In [5]:
# 1. Todos los diagnósticos deben ser K80.0 a K80.8 (sin punto: K800 a K808)
fuera_de_rango = (~datos["DIAG1"].str.match(r"^K80[0-8]$")).sum()
print("Filas con DIAG1 fuera de K80.0-K80.8:", fuera_de_rango)

# 2. La provincia debe quedar asignada en el 100% de los casos, y solo con
#    los dos valores esperados
print()
print("Valores de PROVINCIA_RESIDENCIA:", sorted(datos["PROVINCIA_RESIDENCIA"].dropna().unique()))
print("Filas sin provincia asignada:", datos["PROVINCIA_RESIDENCIA"].isna().sum())

# 3. Las columnas de grupo de riesgo y el flag de colecistectomía deben ser
#    estrictamente booleanas (solo True/False al leerlas desde el CSV)
print()
for columna in ["GRUPO_RIESGO_MUJER", "GRUPO_RIESGO_ETNIA", "GRUPO_RIESGO_40_49", "FLAG_COLECISTECTOMIA"]:
    valores = sorted(datos[columna].dropna().unique())
    print(f"{columna}: {valores}")

Filas con DIAG1 fuera de K80.0-K80.8: 0

Valores de PROVINCIA_RESIDENCIA: ['Cautín', 'Malleco']
Filas sin provincia asignada: 0

GRUPO_RIESGO_MUJER: ['False', 'True']
GRUPO_RIESGO_ETNIA: ['False', 'True']
GRUPO_RIESGO_40_49: ['False', 'True']
FLAG_COLECISTECTOMIA: ['False', 'True']


In [6]:
# 4. Verificación cruzada del flag de colecistectomía: se vuelve a calcular
#    de forma independiente y se compara con la columna ya guardada, para
#    asegurar que la lógica del notebook anterior se aplicó correctamente.
recalculado = datos["GLOSA_INTERV_Q_PPAL"].str.contains("colecistectom", case=False, na=False)
coincide = (recalculado.astype(str) == datos["FLAG_COLECISTECTOMIA"]).all()

print("¿El flag de colecistectomía coincide al recalcularlo?", coincide)

# 5. CONDICION_EGRESO solo debe tomar los códigos documentados por el DEIS: 1 = vivo, 2 = fallecido
print("Valores de CONDICION_EGRESO:", sorted(datos["CONDICION_EGRESO"].unique()))

¿El flag de colecistectomía coincide al recalcularlo? True
Valores de CONDICION_EGRESO: [np.int64(1), np.int64(2)]


## Paso 3 · Filas duplicadas

El validador marca duplicados exactos como un `AVISO`, no como un error que descalifique el dataset. En este tipo de egresos hospitalarios anonimizados, dos personas distintas pueden compartir exactamente las mismas características registradas (sexo, grupo etario, comuna, diagnóstico, etc.), por lo que una fila duplicada no es necesariamente un error de carga: puede representar a dos pacientes reales con el mismo perfil. Esto ya está documentado como limitación en el `README.md` del proyecto para el dataset completo, y aquí se confirma que el subconjunto filtrado mantiene el mismo patrón.

In [7]:
duplicadas = datos.duplicated().sum()
print("Filas duplicadas exactas:", duplicadas, "de", len(datos), "filas totales")
print("Porcentaje:", round(duplicadas / len(datos) * 100, 1), "%")

Filas duplicadas exactas: 1276 de 3789 filas totales
Porcentaje: 33.7 %


## Paso 4 · Guardar el informe de validación

In [8]:
vd.escribir_informe(SALIDA_INFORME, perfil, chequeo, apto, ENTRADA, datos)


Informe escrito en: D:\Sebiche\Magister Unab\Proyecto 1\proyecto-grupo2-mcdi500\docs\validacion_colelitiasis.md


## Resultado

- El subconjunto final (3789 filas, 23 columnas) **cumple** los requisitos mínimos del curso para un dataset multivariado.
- Los 2 avisos (porcentaje de nulos en columnas que ya eran mayoritariamente vacías en el dataset original, y filas duplicadas) son consistentes con lo ya documentado para el dataset completo y no afectan la validez del subconjunto.
- Las validaciones específicas de la transformación no encontraron problemas: todos los diagnósticos quedaron dentro de K80.0-K80.8, el 100% de las filas tiene provincia asignada, las columnas de grupo de riesgo y el flag de colecistectomía son estrictamente booleanas, y el flag de colecistectomía coincide al recalcularlo de forma independiente.
- Informe guardado en `docs/validacion_colelitiasis.md`.

**Para reproducir:** ejecutar en orden `F2_01_obtencion.ipynb`, `F2_03_transformacion.ipynb` y luego este notebook.